In [1]:
import numpy as np
from numpy.linalg import inv
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

from sklearn.datasets import make_spd_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

%matplotlib inline

In [2]:
def is_pos_def(A):
    if is_symmetric(A):
        try:
            np.linalg.cholesky(A)
            return True
        except np.linalg.LinAlgError:
            return False
    else:
        return False

def is_symmetric(a, rtol=1e-05, atol=1e-08):
    return np.allclose(a, a.T, rtol=rtol, atol=atol)

In [3]:
dim = 4
M = 2

In [4]:
G_s = []
sigmas = np.random.rand(M)
covar = np.zeros((dim, dim))
for i in range(M):
    mat = make_spd_matrix(dim)
    G_s.append(mat)
    covar += sigmas[i]*mat
G_s = np.array(G_s)
covar

array([[ 0.04466615,  0.02137266, -0.03020919,  0.00174121],
       [ 0.02137266,  0.20867019, -0.02976112,  0.0075087 ],
       [-0.03020919, -0.02976112,  0.07538664,  0.00450779],
       [ 0.00174121,  0.0075087 ,  0.00450779,  0.03293882]])

In [5]:
H_s = []
prec_coeffs = np.random.rand(M)
precision = np.zeros((dim, dim))
for i in range(M):
    mat = make_spd_matrix(dim)
    H_s.append(mat)
    precision += prec_coeffs[i]*mat
H_s = np.array(H_s)
precision

array([[ 1.09779664, -0.53083702,  0.3195379 ,  0.35887515],
       [-0.53083702,  1.83770669,  0.5972214 , -0.81247611],
       [ 0.3195379 ,  0.5972214 ,  2.61359679, -0.42107834],
       [ 0.35887515, -0.81247611, -0.42107834,  1.11443419]])

In [6]:
is_pos_def(G_s[0]), is_symmetric(G_s[0]), is_pos_def(G_s[1]), is_symmetric(G_s[1])

(True, True, True, True)

In [7]:
is_pos_def(H_s[0]), is_symmetric(H_s[0]), is_pos_def(H_s[1]), is_symmetric(H_s[1])

(True, True, True, True)

In [8]:
sigmas

array([0.01839619, 0.03934877])

In [9]:
prec_coeffs

array([0.63981389, 0.42232927])

In [10]:
covar

array([[ 0.04466615,  0.02137266, -0.03020919,  0.00174121],
       [ 0.02137266,  0.20867019, -0.02976112,  0.0075087 ],
       [-0.03020919, -0.02976112,  0.07538664,  0.00450779],
       [ 0.00174121,  0.0075087 ,  0.00450779,  0.03293882]])

In [11]:
covar.shape

(4, 4)

In [12]:
is_symmetric(covar), is_pos_def(covar)

(True, True)

In [13]:
is_symmetric(precision), is_pos_def(precision)

(True, True)

In [14]:
N = 200
data_sim = np.random.multivariate_normal(np.zeros(dim), covar, N).T
C = np.cov(data_sim)
C, covar

(array([[ 0.04819905,  0.00806335, -0.03656321, -0.00086827],
        [ 0.00806335,  0.19211497, -0.02508191,  0.0084417 ],
        [-0.03656321, -0.02508191,  0.08019409,  0.00404751],
        [-0.00086827,  0.0084417 ,  0.00404751,  0.03402716]]),
 array([[ 0.04466615,  0.02137266, -0.03020919,  0.00174121],
        [ 0.02137266,  0.20867019, -0.02976112,  0.0075087 ],
        [-0.03020919, -0.02976112,  0.07538664,  0.00450779],
        [ 0.00174121,  0.0075087 ,  0.00450779,  0.03293882]]))

In [15]:
def calc_matrices(sigma_hat, G_s, C, dim):
    A = np.zeros((dim, dim))
    B = np.zeros((dim,))
    inv_sigma_hat = inv(sigma_hat) # calculate sigma_hat inverse
    sighat_c_sighat = (inv_sigma_hat.dot(C)).dot(inv_sigma_hat) # calculate sighat*C*sighat
    inv_calc = sighat_c_sighat-inv_sigma_hat # sighat_c_sighat - inv_sighat
    sub_calc = 2*(inv_sigma_hat.dot(C)).dot(inv_sigma_hat)-inv_sigma_hat # 2*inv_calc - sighat
    for h in range(dim):
        for g in range(dim):
            mult = ((inv_sigma_hat.dot(G_s[g])).dot(sub_calc)).dot(G_s[h])
            lhs = np.trace(mult)
            A[g, h] = lhs # because trace, the off diagonals are equal at i,j => j,i ?
        rhs = np.trace(inv_calc.dot(G_s[h]))
        B[h] = rhs
        
    return A, B

def iterative_soln(sig_zero, G_s, C, dim, iters=5):
    sig_imo = sig_zero
    for it in range(iters):
        sigma_hat = (sig_imo.reshape(-1, 1, 1)*G_s).sum(0) # calculate sigma_hat
        A, B = calc_matrices(sigma_hat, G_s, C, dim)
        r_i = inv(A).dot(B)
        sig_i = sig_imo + r_i
        print(sig_i)
        sig_imo = sig_i
    
    return sig_i

def unbiased_init(C, G_s, N, dim):
    A = np.zeros((dim, dim))
    B = np.zeros((dim, ))
    for g in range(dim):
        for h in range(dim):
            A[g, h] = np.trace(G_s[g].dot(G_s[h]))
        B[g] = (N/(N-1))*np.trace(C.dot(G_s[g]))
    return inv(A).dot(B)

def calc_likelihood(sigma, G_s, C, N, P):
    cov_hat = (sigma.reshape(-1, 1, 1)*G_s).sum(0)
    log_l = -P*np.log(2*np.pi) - np.log(np.linalg.det(cov_hat)) - np.trace(np.dot(inv(cov_hat), C))
    
    return log_l*(N/2)

def calc_likelihood_torch(sigma, G_s, C, N, P):
    C_t = torch.tensor(C)
    cov_hat = (sigma.reshape(-1, 1, 1)*G_s).sum(0)
    log_l = -P*np.log(2*np.pi) - torch.log(torch.linalg.det(cov_hat)) - torch.trace(
        torch.matmul(torch.linalg.inv(cov_hat), C_t))
    
    return log_l*(N/2)

In [16]:
sig_zero = unbiased_init(C=C, G_s=G_s, N=N, dim=M)
print(sig_zero, "\n")
sig_i = iterative_soln(sig_zero=sig_zero, G_s=G_s, C=C, dim=M, iters=20)

[0.02003481 0.03521284] 

[0.02179754 0.03581301]
[0.02207566 0.03579688]
[0.02208054 0.03579571]
[0.02208054 0.03579571]
[0.02208054 0.03579571]
[0.02208054 0.03579571]
[0.02208054 0.03579571]
[0.02208054 0.03579571]
[0.02208054 0.03579571]
[0.02208054 0.03579571]
[0.02208054 0.03579571]
[0.02208054 0.03579571]
[0.02208054 0.03579571]
[0.02208054 0.03579571]
[0.02208054 0.03579571]
[0.02208054 0.03579571]
[0.02208054 0.03579571]
[0.02208054 0.03579571]
[0.02208054 0.03579571]
[0.02208054 0.03579571]


In [17]:
sig_i, sigmas

(array([0.02208054, 0.03579571]), array([0.01839619, 0.03934877]))

In [18]:
cov_hat = (sig_zero.reshape(-1, 1, 1)*G_s).sum(0)
cov_hat, covar

(array([[ 0.04228701,  0.01614759, -0.02839455,  0.00125138],
        [ 0.01614759,  0.19999053, -0.02160938,  0.00730764],
        [-0.02839455, -0.02160938,  0.07112831,  0.00422994],
        [ 0.00125138,  0.00730764,  0.00422994,  0.03209041]]),
 array([[ 0.04466615,  0.02137266, -0.03020919,  0.00174121],
        [ 0.02137266,  0.20867019, -0.02976112,  0.0075087 ],
        [-0.03020919, -0.02976112,  0.07538664,  0.00450779],
        [ 0.00174121,  0.0075087 ,  0.00450779,  0.03293882]]))

In [19]:
sum((sig_i-sigmas)**2), sum((sig_zero-sigmas)**2)

(2.6198694871410265e-05, 1.9791032722933308e-05)

In [20]:
calc_likelihood(sig_zero, G_s, C, N=N, P=dim)

-31.733195921449564

In [21]:
calc_likelihood(sig_i, G_s, C, N=N, P=dim)

-31.152897981647065

In [71]:
"""
PRECISION
"""

N = 200
data_sim = np.random.multivariate_normal(np.zeros(dim), inv(precision), N).T
C = np.cov(data_sim)
C, inv(precision)

(array([[ 1.40073271,  0.25620852, -0.27243427, -0.38412284],
        [ 0.25620852,  0.85478499, -0.14647102,  0.48952843],
        [-0.27243427, -0.14647102,  0.47417454,  0.13601675],
        [-0.38412284,  0.48952843,  0.13601675,  1.33088401]]),
 array([[ 1.23201601,  0.32860911, -0.26730863, -0.25816756],
        [ 0.32860911,  0.91337564, -0.16893701,  0.49624332],
        [-0.26730863, -0.16893701,  0.47695857,  0.14313099],
        [-0.25816756,  0.49624332,  0.14313099,  1.39631853]]))

In [90]:
def calc_matrices_precision(psi_hat, H_s, C, dim):
    A = np.zeros((dim, dim))
    B = np.zeros((dim,))
    inv_psi_hat = inv(psi_hat) # calculate psi_hat inverse
    for g in range(dim):
        for h in range(dim):
            mult = ((inv_psi_hat.dot(H_s[g])).dot(inv_psi_hat)).dot(H_s[h])
            lhs = np.trace(mult)
            A[g, h] = lhs
        rhs = np.trace(inv_psi_hat.dot(H_s[g])) - np.trace(C.dot(H_s[g]))
        B[g] = rhs
        
    return A, B

def iterative_soln_precision(coeffs_zero, H_s, C, dim, iters=5):
    s_imo = coeffs_zero
    for it in range(iters):
        psi_hat = (s_imo.reshape(-1, 1, 1)*H_s).sum(0) # calculate psi_hat
        A, B = calc_matrices_precision(psi_hat, H_s, C, dim)
        t_i = inv(A).dot(B)
        s_i = s_imo + t_i
        print(s_i)
        s_imo = s_i
        
    return s_imo

def unbiased_init_precision(C, H_s, N, dim):
    A = np.zeros((dim, dim))
    B = np.zeros((dim, ))
    for g in range(dim):
        for h in range(dim):
            A[g, h] = np.trace(H_s[g].dot(H_s[h]))
        B[g] = np.trace(inv(C).dot(H_s[g]))
    return inv(A).dot(B)

def calc_likelihood_precision(coeffs, H_s, C, N, P):
    precision_hat = (coeffs.reshape(-1, 1, 1)*H_s).sum(0)
    log_l = -P*np.log(2*np.pi) + np.log(np.linalg.det(precision_hat)) - np.trace(np.dot(precision_hat, C))
    
    return log_l*(N/2)

In [91]:
calc_matrices_precision(inv(C), H_s, C, dim=M)

(array([[4.64990426, 2.33619934],
        [2.33619934, 5.30533266]]),
 array([0., 0.]))

In [92]:
s_zero = unbiased_init_precision(C=C, H_s=H_s, N=N, dim=M)
s_zero, prec_coeffs

(array([0.66161999, 0.40887919]), array([0.63981389, 0.42232927]))

In [93]:
calc_likelihood_precision(s_zero, H_s, C, N, P=dim)

-1038.6703562006485

In [94]:
coeffs_hat = iterative_soln_precision(s_zero, H_s, C, dim=M, iters=15)
coeffs_hat

[0.64722733 0.40732532]
[0.64751078 0.40730517]
[0.64751089 0.40730516]
[0.64751089 0.40730516]
[0.64751089 0.40730516]
[0.64751089 0.40730516]
[0.64751089 0.40730516]
[0.64751089 0.40730516]
[0.64751089 0.40730516]
[0.64751089 0.40730516]
[0.64751089 0.40730516]
[0.64751089 0.40730516]
[0.64751089 0.40730516]
[0.64751089 0.40730516]
[0.64751089 0.40730516]


array([0.64751089, 0.40730516])

In [95]:
psi_hat = (s_zero.reshape(-1, 1, 1)*H_s).sum(0) # calculate psi_hat
psi_hat

array([[ 1.09827013, -0.53089328,  0.29014537,  0.35954556],
       [-0.53089328,  1.87351468,  0.64137561, -0.83796283],
       [ 0.29014537,  0.64137561,  2.61328962, -0.45550855],
       [ 0.35954556, -0.83796283, -0.45550855,  1.14235089]])

In [96]:
precision

array([[ 1.09779664, -0.53083702,  0.3195379 ,  0.35887515],
       [-0.53083702,  1.83770669,  0.5972214 , -0.81247611],
       [ 0.3195379 ,  0.5972214 ,  2.61359679, -0.42107834],
       [ 0.35887515, -0.81247611, -0.42107834,  1.11443419]])

In [67]:
calc_likelihood_torch(torch.tensor(sig_i), torch.tensor(G_s), C, N=N, P=dim)

tensor(-1199.7531, dtype=torch.float64)

In [91]:
G_s_t = torch.nn.Parameter(torch.tensor(G_s))
calc_likelihood_torch(torch.tensor(sig_i), G_s_t, C, N=N, P=dim)

tensor(-1199.7531, dtype=torch.float64, grad_fn=<MulBackward0>)

In [94]:
optimizer = optim.Adam([G_s_t], lr=1e-3, betas=(0.9, 0.999))
for ep in range(100):
    optimizer.zero_grad()
    loss = -calc_likelihood_torch(torch.tensor(sig_i), G_s_t, C, N=N, P=dim)
    print(loss.item())
    loss.backward()
    optimizer.step()

1199.4956571932098
1199.4504742240433
1199.4074798284087
1199.3666715063507
1199.3280315521706
1199.2915234388806
1199.2570876769485
1199.2246378605373
1199.1940584126723
1199.1652061566506
1199.1379172831073
1199.1120189455632
1199.0873419684838
1199.063730654465
1199.0410479496084
1199.019176819879
1198.9980195870257
1198.9774964746762
1198.9575438875568
1198.9381125585655
1198.9191656087396
1198.900676582666
1198.8826275358545
1198.8650072384958
1198.8478095353096
1198.8310318786876
1198.814674038269
1198.7987369841235
1198.7832219398902
1198.768129603538
1198.7534595348222
1198.7392097088086
1198.7253762335197
1198.711953226665
1198.698932841745
1198.68630542794
1198.6740598018298
1198.6621836030374
1198.6506637013395
1198.6394866206163
1198.628638945822
1198.6181076830967
1198.607880549801
1198.5979461797297
1198.588294237914
1198.5789154480372
1198.5698015426551
1198.5609451515006
1198.552339646022
1198.543978959046
1198.535857397491
1198.5279694638127
1198.520309698784
1198.5128

In [95]:
G_s_t

Parameter containing:
tensor([[[ 1.3435,  1.5853, -0.2613, -0.3635],
         [ 1.5853,  3.4699, -0.2076, -0.8714],
         [-0.2613, -0.2076,  0.8111,  0.0967],
         [-0.3635, -0.8714,  0.0967,  1.1172]],

        [[ 1.8685, -0.1725,  1.7359, -0.0403],
         [-0.1725,  0.7322,  0.0672,  0.1507],
         [ 1.7359,  0.0672,  3.4956, -0.2751],
         [-0.0403,  0.1507, -0.2751,  0.7514]]], dtype=torch.float64,
       requires_grad=True)

In [96]:
G_s

array([[[ 1.4103558 ,  1.66301157, -0.1892549 , -0.40579989],
        [ 1.66301157,  3.5609808 , -0.25671097, -0.8619859 ],
        [-0.1892549 , -0.25671097,  0.90613756,  0.11264728],
        [-0.40579989, -0.8619859 ,  0.11264728,  1.05080232]],

       [[ 1.93541998, -0.09481825,  1.80791025, -0.08268352],
        [-0.09481825,  0.82330029,  0.01813595,  0.16010977],
        [ 1.80791025,  0.01813595,  3.59066133, -0.25913208],
        [-0.08268352,  0.16010977, -0.25913208,  0.68501295]]])